# L27 · 不确定中找规律 — 概率基础

**目标**：用 numpy 模拟随机事件，观察「样本多了，频率趋近概率」。

**为什么对 Aurora 重要**：`np.random.default_rng(seed)` 贯穿训练全流程——batch 采样用它洗牌、dropout 用它生成掩码、数据增强用它选变换参数；seed 固定后结果可复现，调参和 debug 才有意义。

## 本课剧情：在随机里等秩序浮现

一次掷骰子很任性，一万次掷骰子就开始讲道理。这节课实现 `estimate_prob_six(n, seed)`：固定 `seed` 保证每次运行结果一致，逐步增大 `n`，观察频率如何向 1/6 收敛。

## 1. 用固定种子的随机数发生器（保证可复现）

`np.random.default_rng(seed)` 在内部维护独立的随机状态对象（PCG64 算法），与全局 `np.random.seed()` 完全隔离：调用它不会影响其他模块的随机序列，多个 `rng` 实例之间互不干扰。这是 numpy 1.17+ 推荐的现代用法。

`rng.integers(1, 7, size=n)` 一次性生成 n 个随机整数，底层是 C 实现的批量采样；相比 Python 层面的 `for` 循环逐个生成，速度快 100 倍以上（n=10000 时约 50μs vs 5ms）。

## 实验入口：概率量如何从样本里出现

注意 `size` 和 `seed` 两个参数：`size` 决定估计精度，`seed` 决定结果是否可复现。运行下面的格，观察 `flips.mean()` 是否接近 0.5。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
flips = rng.integers(0, 2, size=1000)   # 0/1 当正反面
print('1000 次抛硬币，正面比例 ≈', flips.mean(), ' (应接近 0.5)')

## 动手观察：小样本 vs 大样本

下面对比 `n=10`、`100`、`10000` 时掷骰子的频率估计，观察样本量增大时估计值如何向理论值 0.167 靠拢。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    print(f'n={n:5d}  估计 P(掷到6) = {estimate:.3f}')


## 代码实验：多次重复同一个随机实验

用 5 个不同 `seed` 重复同一个 `n` 的实验：`n=2000` 时各次估计值聚集在 0.167 附近，`n=20` 时则明显分散。

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        rolls = rng.integers(1, 7, size=n)
        estimates.append(np.mean(rolls == 6))
    print(f'n={n:4d} ->', np.round(estimates, 3), '平均=', round(float(np.mean(estimates)), 3))


## 2. ✏️ 实现 `estimate_prob_six(n, seed=0)`

模拟掷 n 次骰子(1~6)，返回出现 6 的频率。

**推理路线**：
1. 创建独立随机状态：`rng = np.random.default_rng(seed)`，不污染全局随机序列。
2. 一次性生成 n 次投掷：`rolls = rng.integers(1, 7, size=n)`，区间是 `[1, 7)`，即 1~6。
3. 统计 6 的频率：`np.mean(rolls == 6)` 等价于 `sum(rolls == 6) / n`，返回 0~1 的浮点数。

**参考输入输出**：n=6, seed=0 → 可能得到 `[4,5,6,6,1,3]`，频率=2/6≈0.333；n=10000, seed=0 → 频率会收敛到约 0.166。

<details><summary>点击查看参考实现</summary>

```python
def estimate_prob_six(n, seed=0):
    rng = np.random.default_rng(seed)
    rolls = rng.integers(1, 7, size=n)
    return np.mean(rolls == 6)
```

</details>

### 写代码前，先把变量表补完整

写 `estimate_prob_six` 前明确三件事：
- 输入：`n`（掷骰次数）和 `seed`（随机种子，默认 0）
- 关键步骤：用 `rng.integers(1, 7, size=n)` 生成样本，对 `sample == 6` 取均值
- 返回：一个浮点数，表示 6 出现的频率，`n=200000` 时应在 0.165–0.169 之间

In [ ]:
def estimate_prob_six(n, seed=0):
    # ✏️ TODO: 返回掷 n 次骰子中 6 出现的频率
    return None  # ← 改这里


In [ ]:
p = estimate_prob_six(200_000)
print('P(掷出6) ≈', round(p, 4), ' (真值 1/6 ≈ 0.1667)')
assert abs(p - 1/6) < 0.01
print('✅ 通过：样本够多，频率逼近真实概率。')

**🔗 Aurora 连接**：`src/aurora/audio/` 中的数据增强管道和训练循环都通过 `np.random.default_rng(seed)` 生成 batch 洗牌索引和 dropout 掩码。固定 seed 后，两次运行产生完全一致的 batch 序列——这是调参和 debug 时对照实验有意义的前提。

In [ ]:
for n in [20, 100, 1000, 10000]:
    rng = np.random.default_rng(7)
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    error = abs(estimate - 1/6)
    print(f'n={n:5d} | 估计={estimate:.4f} | 离理论值差={error:.4f}')


## 参数实验：只改一个旋钮

固定 `seed=42`，把 `n` 从 10 增加到 10000，每次打印频率。观察 n=100 时波动约 ±0.05，n=10000 时波动约 ±0.005（标准差 ∝ 1/√n）。

In [ ]:
for n in [20, 100, 1000, 10000]:
    rng = np.random.default_rng(7)
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    error = abs(estimate - 1/6)
    print(f'n={n:5d} | 估计={estimate:.4f} | 离理论值差={error:.4f}')


## 本课收束

现在可以用 `estimate_prob_six(n, seed)` 返回掷骰频率，`n=200000` 时估计值稳定在 0.1667 以内。Aurora 训练循环中的 batch 洗牌和 dropout mask 生成都依赖相同的 `default_rng` 接口，固定 seed 是让实验可复现的最小代价。下一节将用均值和方差刻画样本分布的形状，今天的 `np.mean(sample == 6)` 是样本均值的特例。频率随 `n` 增大收敛的速率是 1/√n，这也是下节引入标准差时的核心关系。

In [ ]:
# 小检查：同一个实验，样本量越大越稳定
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.integers(1, 7, size=n)
    estimate = np.mean(samples == 6)
    print(f'n={n:4d} -> P(6)≈{estimate:.3f}')
